In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import plotly.express as px
import joblib
from sklearn.metrics import silhouette_score

###  Calculate RFM metrics

In [2]:

def calculate_rfm(df):
    """Calculate RFM metrics"""
    latest_date = df['InvoiceDate'].max()
    
    rfm = df.groupby('Customer ID').agg({
        'InvoiceDate': lambda x: (latest_date - x.max()).days,   # Recency
        'Invoice': 'nunique',                                    # Frequency
        'TotalAmount': 'sum'                                     # Monetary
    }).reset_index()
    
    rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']
    return rfm

### Segmentaion by k means clutering

In [3]:
def perform_segmentation(rfm, n_clusters=5):
    """Perform K-Means Clustering"""
    features = rfm[['Recency', 'Frequency', 'Monetary']]
    
    scaler = StandardScaler()
    scaled_features = scaler.fit_transform(features)
    
    # KMeans
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    rfm['Cluster'] = kmeans.fit_predict(scaled_features)
    
    # Segment Labels
    cluster_labels = {
        0: "At Risk",
        1: "VIP / Champions",
        2: "New Customers",
        3: "Loyal Customers",
        4: "Hibernating"
    }
    rfm['Segment'] = rfm['Cluster'].map(cluster_labels)
    
    # Save model
    joblib.dump(kmeans, 'kmeans_model.pkl')
    joblib.dump(scaler, 'scaler.pkl')
    
    return rfm, kmeans, scaler

In [4]:
def evaluate_clustering(rfm):
    """Evaluate clustering quality"""
    features = rfm[['Recency', 'Frequency', 'Monetary']]
    scaler = StandardScaler()
    scaled = scaler.fit_transform(features)
    score = silhouette_score(scaled, rfm['Cluster'])
    return score

In [5]:

def get_segment_summary(df):
    """Get summary by segment"""
    summary = df.groupby('Segment_Name').agg({
        'Customer ID': 'count',
        'Purchase Amount (USD)': ['mean', 'sum'],
        'Previous Purchases': 'mean',
        'Review Rating': 'mean',
        'Age': 'mean'
    }).round(2)
    
    summary.columns = ['Customer Count', 'Avg Purchase (USD)', 'Total Revenue', 
                       'Avg Previous Purchases', 'Avg Rating', 'Avg Age']
    return summary